## Whole notebook

In [1]:
from pathlib import Path
from trail_pacer.utils import DATA_PATH

GPX_FILE = Path(DATA_PATH,"gpx", "TSB2026-LA_BOUCLE_CUGEOISE.gpx")

In [2]:
from trail_pacer.utils import gpx_to_dataframe

df = gpx_to_dataframe(GPX_FILE)
print(f"Track points : {len(df)}")
print(f"Total distance : {df['distance_m'].max() / 1000:.2f} km")
print(f"Min elevation : {df['elevation_m'].min():.0f} m")
print(f"Max elevation : {df['elevation_m'].max():.0f} m")
df.head()

Track points : 1218
Total distance : 25.86 km
Min elevation : 185 m
Max elevation : 1035 m


,lat,lon,elevation_m,distance_m
0,43.279237,5.703046,223.24,0.000000
1,43.279237,5.703046,222.66,0.006785
2,43.278894,5.703307,222.26,43.644764
3,43.278930,5.703258,221.61,49.344395
4,43.278837,5.703127,220.66,64.121342


In [3]:
from trail_pacer.utils import interpolate_splits, compute_split_stats

In [4]:
splits = interpolate_splits(df, step_km=0.1)
splits

,distance_km,elevation_m
0,0.000000,223.240000
1,0.100000,218.144603
2,0.200000,208.730788
3,0.300000,204.957440
4,0.400000,199.990665
...,...,...
255,25.500000,248.584702
256,25.600000,236.987459
257,25.700000,227.977752
258,25.800000,224.290136


In [5]:
split_stats = compute_split_stats(splits)
split_stats

,distance_km,elevation_m,elev_diff_m,split_gain_m,split_loss_m,cum_gain_m,cum_loss_m,grade_pct
0,0.000000,223.240000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
1,0.100000,218.144603,-5.095397,0.0,5.095397,0.000000,5.095397,-5.095397
2,0.200000,208.730788,-9.413815,0.0,9.413815,0.000000,14.509212,-9.413815
3,0.300000,204.957440,-3.773347,0.0,3.773347,0.000000,18.282560,-3.773347
4,0.400000,199.990665,-4.966775,0.0,4.966775,0.000000,23.249335,-4.966775
...,...,...,...,...,...,...,...,...
255,25.500000,248.584702,-5.300703,0.0,5.300703,1612.409918,1587.065215,-5.300703
256,25.600000,236.987459,-11.597243,0.0,11.597243,1612.409918,1598.662459,-11.597243
257,25.700000,227.977752,-9.009706,0.0,9.009706,1612.409918,1607.672165,-9.009706
258,25.800000,224.290136,-3.687617,0.0,3.687617,1612.409918,1611.359782,-3.687617


### 🏃‍♂️ Riegel Formula for Predicting Race Times

The **Riegel formula** is a simple model used to estimate how long a runner will take to complete a race of a different distance based on a known performance. It assumes that performance decreases at a predictable rate as distance increases.

The formula is:

$
T_2 = T_1 \left(\frac{D_2}{D_1}\right)^{1.06}
$

- $T_1$: known time  
- $D_1$: known distance  
- $T_2$: predicted time  
- $D_2$: target distance  
- **1.06**: fatigue exponent (typical for trained runners)

A higher exponent means the runner slows down more as distance increases. The model works well for most endurance events from 1500 m to the marathon.


In [6]:
from trail_pacer.utils import Conversions as conv

def base_pace_from_flat_times(num_kms,perf: tuple,fatigue_rate: float) -> float:
    RIEGEL_EXP   = 1+fatigue_rate

    goal_km, time_min = perf
    TIME_MIN = conv.chrono_to_min(time_min)
    norm_time = TIME_MIN * (num_kms / goal_km) ** RIEGEL_EXP
    return float(norm_time)

# flat_perf = (5,   "18:35")
# flat_perf = (5,   "23:30")
flat_perf = (10,  "38:51")
# flat_perf = (10,  "49:00")
# flat_perf = (21.1,"1:23:00")
# flat_perf = (21.1,"1:48:00")

num_kms = 26
base_pace_min_per_km = base_pace_from_flat_times(num_kms=num_kms, perf=flat_perf,fatigue_rate=0.06)
print(f"Time for {num_kms} km: "+conv.min_to_chrono(base_pace_min_per_km))

Time for 26 km: 1:46:58


In [7]:
conv.chrono_to_min("38:51")

38.85

In [8]:
conv.chrono_to_min("38:51")/10

3.8850000000000002

In [9]:
conv.min_to_chrono(conv.chrono_to_min("38:51")/10)

'3:53'

In [ ]:

import numpy as np
import pandas as pd
from datetime import datetime


In [11]:
from datetime import datetime

model = PaceModel(pace_10k_min_per_km='3:53', up_sens=0.8, down_sens=0.02, fatigue_rate=0.00)

model.summary(split_stats)

splits = model.predict_split_times(split_stats, start_time=datetime(2025, 6, 1, 8, 0))
splits["cum_predicted_pace"] = splits["predicted_pace"].cumsum()
splits[["distance_km", "elevation_m", "grade_pct", "predicted_pace", "cum_time_str", "arrival_time"]]

Base pace       : 3.88 min/km
Distance        : 25.9 km
Flat estimate   : 1:40:24
Grade-adjusted  : 3:46:10
Grade + fatigue : 3:46:10


,distance_km,elevation_m,grade_pct,predicted_pace,cum_time_str,arrival_time
0,0.000000,223.240000,0.000000,3.883333,0:00:00,2025-06-01 08:00:00.000000000
1,0.100000,218.144603,-5.095397,3.781425,0:00:23,2025-06-01 08:00:22.688552412
2,0.200000,208.730788,-9.413815,3.695057,0:00:45,2025-06-01 08:00:44.858894550
3,0.300000,204.957440,-3.773347,3.807866,0:01:08,2025-06-01 08:01:07.706092854
4,0.400000,199.990665,-4.966775,3.783998,0:01:30,2025-06-01 08:01:30.410079846
...,...,...,...,...,...,...
255,25.500000,248.584702,-5.300703,3.777319,3:44:51,2025-06-01 11:44:50.619779406
256,25.600000,236.987459,-11.597243,3.651388,3:45:13,2025-06-01 11:45:12.528110196
257,25.700000,227.977752,-9.009706,3.703139,3:45:35,2025-06-01 11:45:34.746945420
258,25.800000,224.290136,-3.687617,3.809581,3:45:58,2025-06-01 11:45:57.604431384


# Comparing Predictions with actual times from Pierre's run

In [12]:
import pandas as pd

df_temps_pierre = pd.read_excel("/Users/sauniere/Desktop/Perso/Codes/Running/trail-pacer/data/Temps_Pierre_Cugeoise.xlsx")
for col in ['Time','GAP /km']:
    df_temps_pierre[col] = pd.to_datetime(df_temps_pierre[col], format="%H:%M:%S")
    df_temps_pierre[col] = df_temps_pierre[col].dt.strftime("%H:%M")

df_temps_pierre["pace"] = df_temps_pierre["Time"].apply(conv.chrono_to_min) 
# Add a column with the cumulated pace up to that point
df_temps_pierre["cum_pace"] = df_temps_pierre["pace"].cumsum()
df_temps_pierre["cum_dist"] = df_temps_pierre["Distance km"].cumsum()
df_temps_pierre.head(2)

,Lap,Distance km,Time,GAP /km,Elev,HR,pace,cum_pace,cum_dist
0,1,1.0,05:13,05:16,-22,146,5.216667,5.216667,1.0
1,2,1.0,07:39,04:48,80,157,7.650000,12.866667,2.0


In [13]:
splits.head(2)

,distance_km,elevation_m,elev_diff_m,split_gain_m,split_loss_m,cum_gain_m,cum_loss_m,grade_pct,predicted_pace,split_time_min,cum_time_min,cum_time_str,arrival_time,cum_predicted_pace
0,0.0,223.240000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,3.883333,0.000000,0.000000,0:00:00,2025-06-01 08:00:00.000000000,3.883333
1,0.1,218.144603,-5.095397,0.0,5.095397,0.0,5.095397,-5.095397,3.781425,0.378143,0.378143,0:00:23,2025-06-01 08:00:22.688552412,7.664759


In [14]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["elevation_m"],
        fill="tozeroy", fillcolor="rgba(100,150,255,0.15)",
        line=dict(color="rgba(100,150,255,0.6)", width=1.5),
        name="Elevation (m)",
        hovertemplate="%{y:.0f} m<extra></extra>",
    ),
    secondary_y=False,
)

splits["pace_fmt"]    = splits["predicted_pace"].apply(conv.min_to_chrono)

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["predicted_pace"],
        line=dict(color="tomato", width=2.5),
        name="Predicted Pace",
        customdata=splits["pace_fmt"],
        hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=df_temps_pierre["cum_dist"], y=df_temps_pierre["pace"],
        line=dict(color="darkorange", width=2, dash="dot"),
        name="Real Pace",
        # customdata=df_temps_pierre["fatigue_fmt"],
        # hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Elevation & Predicted Pace Profile",
    xaxis_title="Cumulative Distance (km)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.update_yaxes(title_text="Elevation (m)",    secondary_y=False)
fig.update_yaxes(title_text="Pace (min/km)",    secondary_y=True,  autorange="reversed")

fig.show()
# fig.write_html("data/chart_elevation_pace.html", include_plotlyjs="cdn", full_html=False)
# print("✅ Embeddable div saved → data/chart_elevation_pace.html")

In [18]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["elevation_m"],
        fill="tozeroy", fillcolor="rgba(100,150,255,0.15)",
        line=dict(color="rgba(100,150,255,0.6)", width=1.5),
        name="Elevation (m)",
        hovertemplate="%{y:.0f} m<extra></extra>",
    ),
    secondary_y=False,
)

splits["pace_fmt"]    = splits["predicted_pace"].apply(conv.min_to_chrono)

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["predicted_pace"],
        line=dict(color="tomato", width=2.5),
        name="Predicted Pace",
        customdata=splits["pace_fmt"],
        hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=df_temps_pierre["cum_dist"]-1, y=df_temps_pierre["pace"],
        line=dict(color="darkorange", width=2, dash="dot"),
        name="Real Pace",
        # customdata=df_temps_pierre["fatigue_fmt"],
        # hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Elevation & Predicted Pace Profile",
    xaxis_title="Cumulative Distance (km)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.update_yaxes(title_text="Elevation (m)",    secondary_y=False)
fig.update_yaxes(title_text="Pace (min/km)",    secondary_y=True,  autorange="reversed")

fig.show()
# fig.write_html("data/chart_elevation_pace.html", include_plotlyjs="cdn", full_html=False)
# print("✅ Embeddable div saved → data/chart_elevation_pace.html")

In [ ]:
# DIST_COL = "distance_km"
LEN_COL = "distance_km"
GAIN_COL = "split_gain_m"
LOSS_COL = "split_loss_m"
GRADE_COL = "grade_pct"
NET_COL = "net_gain_m"

In [ ]:
df = split_stats.copy()

In [30]:
def naismith_time_minutes(
    distance_km: pd.Series,
    gain_m: pd.Series,
    flat_speed_kmh: float = 8.0,   # running flat pace
) -> pd.Series:
    """
    Return estimated split time in minutes using Naismith's Rule.
    One extra hour per 600 m of ascent.
    """
    flat_time_min  = (distance_km / flat_speed_kmh) * 60
    climb_time_min = (gain_m / 600.0) * 60
    return flat_time_min + climb_time_min

df["naismith_min"] = naismith_time_minutes(
    df[LEN_COL], df[GAIN_COL], flat_speed_kmh=8.0
)
df["naismith_pace_min_km"] = df["naismith_min"] / df[LEN_COL]

print(f"Predicted finish time (Naismith): "
      f"{df['naismith_min'].sum() / 60:.2f} h  "
      f"({int(df['naismith_min'].sum() // 60)}h "
      f"{int(df['naismith_min'].sum() % 60)}min)")

Predicted finish time (Naismith): 46.34 h  (46h 20min)


In [ ]:
def minetti_cost(grade_decimal: float) -> float:
    """Energy cost in J/(kg·m) for a given grade (Minetti et al. 2002)."""
    i = grade_decimal
    return (155.4 * i**5 - 30.4 * i**4 - 43.3 * i**3
            + 46.3 * i**2 + 19.5 * i + 3.6)

FLAT_COST = minetti_cost(0.0)   # ≈ 3.6 J/(kg·m)

def minetti_pace_multiplier(grade_pct: pd.Series) -> pd.Series:
    """Pace multiplier vs flat running (>1 = slower, <1 = faster)."""
    grade_dec = grade_pct / 100.0
    cost = grade_dec.apply(minetti_cost)
    return (cost / FLAT_COST).clip(0.5, 4.0)

df["minetti_multiplier"] = minetti_pace_multiplier(df[GRADE_COL])

# Visualise the Minetti curve
grades = np.linspace(-0.35, 0.35, 200)
costs  = [minetti_cost(g) for g in grades]
multipliers = [c / FLAT_COST for c in costs]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(grades * 100, costs, color="steelblue", linewidth=2)
axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("Grade (%)")
axes[0].set_ylabel("Energy cost J/(kg·m)")
axes[0].set_title("Minetti Metabolic Cost Curve")
axes[0].grid(True, linestyle="--", alpha=0.4)

axes[1].plot(grades * 100, multipliers, color="tomato", linewidth=2)
axes[1].axhline(1.0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("Grade (%)")
axes[1].set_ylabel("Pace multiplier (relative to flat)")
axes[1].set_title("Minetti Pace Multiplier")
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("data/minetti_curve.png", dpi=150)
plt.show()